# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
import os
import json
import numpy as np
import pandas as pd

# Work from the repo root whether this runs locally or fresh in Colab
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    os.system("git clone https://github.com/PrathamDudani/FlyRank_Assignment.git")
    os.chdir("FlyRank_Assignment")

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

pd.set_option("display.max_columns", 60)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)


(30000, 44)


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

This playbook reuses the two validated signals from earlier weeks instead of inventing a new
model:

1. **The honest model** (`w05_model.ipynb` / audited in `w06_validation_audit.ipynb`) — a
   six-feature logistic regression, evaluated on a **client-grouped holdout** so no client's
   other pages leak into the score. Audited AUC: 0.542, against a base rate of 0.540 — a real
   but modest signal, not a strong one.
2. **The transparent baseline rule** (`w04_baseline_score.ipynb`) — flags pages whose CTR sits
   meaningfully below the average CTR for their position bucket. No black box: a reviewer can
   recompute this by hand from the position/CTR table.

The queue below is scored **out-of-sample**, on the same client-held-out test split audited in
Week 6 (identical `random_state=42` split) — so every score in this playbook is a score the model
was not trained on, matching how it would actually see a brand-new client's content.


In [10]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

valid = df[df["avg_position"] > 0].copy()
valid["is_declining_label"] = (valid["trend_direction"] == "down").astype(int)

features = ["ctr", "avg_position", "impressions_90d", "engagement_rate",
            "days_since_last_update", "search_volume"]

# Identical split to w06_validation_audit.ipynb -- same seed, same grouping --
# so this playbook's queue is traceable back to that audit's numbers.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(valid, groups=valid["client_id"]))
train_grouped, test_grouped = valid.iloc[train_idx].copy(), valid.iloc[test_idx].copy()

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(train_grouped[features].fillna(0), train_grouped["is_declining_label"])

queue = test_grouped.copy()
queue["model_probability"] = model.predict_proba(queue[features].fillna(0))[:, 1]

auc_check = round(roc_auc_score(queue["is_declining_label"], queue["model_probability"]), 3)
print(f"Re-derived holdout AUC: {auc_check} (should match the 0.542 audited in w06)")


Re-derived holdout AUC: 0.542 (should match the 0.542 audited in w06)


In [11]:
# The transparent baseline rule (from w04): expected CTR by position bucket,
# learned ONLY from the training clients, then applied to the held-out queue --
# so the rule itself doesn't peek at the clients it's scoring, matching the
# grouped-split discipline used for the model.
bucket_bins = [0, 3, 6, 10, 20, 1000]
bucket_labels = ["1-3", "4-6", "7-10", "11-20", "20+"]

train_grouped["position_bucket"] = pd.cut(train_grouped["avg_position"], bins=bucket_bins, labels=bucket_labels).astype(str)
queue["position_bucket"] = pd.cut(queue["avg_position"], bins=bucket_bins, labels=bucket_labels).astype(str)

expected_ctr_by_bucket = train_grouped.groupby("position_bucket")["ctr"].mean().to_dict()
queue["expected_ctr"] = queue["position_bucket"].map(expected_ctr_by_bucket).astype(float)
queue["ctr_gap"] = queue["expected_ctr"] - queue["ctr"]

# Thresholds fit on TRAIN only, applied to the queue -- no peeking.
train_grouped["expected_ctr"] = train_grouped["position_bucket"].map(expected_ctr_by_bucket).astype(float)
train_gap = train_grouped["expected_ctr"] - train_grouped["ctr"]
gap_threshold = train_gap.quantile(0.75)
visibility_threshold = train_grouped["impressions_90d"].median()
staleness_threshold = train_grouped["days_since_last_update"].median()

queue["ctr_gap_flag"] = queue["ctr_gap"] > gap_threshold
queue["is_visible"] = queue["impressions_90d"] >= visibility_threshold
queue["is_stale"] = queue["days_since_last_update"] >= staleness_threshold
queue["model_flag"] = queue["model_probability"] >= 0.60

print("Rule + model flags built on", len(queue), "held-out rows")
print(queue[["ctr_gap_flag", "is_visible", "is_stale", "model_flag"]].mean().round(3))


Rule + model flags built on 5821 held-out rows
ctr_gap_flag    0.379
is_visible      0.407
is_stale        0.259
model_flag      0.030
dtype: float64


In [12]:
# Reason codes + archetype -> action mapping.
# Archetypes here are simple RULE-BASED segments (visible/stale/flagged combinations),
# not statistical clusters -- named that way on purpose so nobody reads them as
# something more rigorous (e.g. k-means) than they are.

def reason_codes(row):
    codes = []
    if row["model_flag"]:
        codes.append("MODEL_FLAGGED_DECLINE_RISK")
    if row["ctr_gap_flag"]:
        codes.append("CTR_BELOW_POSITION_EXPECTED")
    if row["is_visible"] and row["is_stale"]:
        codes.append("VISIBLE_AND_STALE")
    if not row["is_visible"]:
        codes.append("LOW_VISIBILITY")
    return codes or ["NO_FLAG"]


def archetype_and_action(row):
    codes = set(row["reason_codes"])
    if "MODEL_FLAGGED_DECLINE_RISK" in codes and row["is_visible"]:
        return "Visible & Declining", "PRIORITY_REFRESH_REVIEW"
    if "CTR_BELOW_POSITION_EXPECTED" in codes and row["avg_position"] <= 20:
        return "Striking-Distance CTR Gap", "REVIEW_SNIPPET_CTR"
    if "VISIBLE_AND_STALE" in codes:
        return "Visible & Stale", "SCHEDULE_REFRESH"
    if "LOW_VISIBILITY" in codes:
        return "Quiet / Low Visibility", "MONITOR_ONLY"
    return "Stable", "NO_ACTION"


queue["reason_codes"] = queue.apply(reason_codes, axis=1)
queue[["archetype", "suggested_action"]] = queue.apply(
    lambda r: pd.Series(archetype_and_action(r)), axis=1
)
queue["reason_codes_str"] = queue["reason_codes"].apply(lambda c: "|".join(c))

# Simple, disclosed confidence label -- not a probability, just a triage tier.
def confidence(row):
    if row["model_flag"] and row["is_visible"]:
        return "high"
    if row["suggested_action"] != "NO_ACTION":
        return "medium"
    return "low"

queue["confidence"] = queue.apply(confidence, axis=1)

# Rank by ARCHETYPE PRIORITY first (the human-facing "what matters most" order),
# then by traffic at stake within each tier. Earlier drafts ranked by raw model
# probability first and it backfired: near-zero-traffic pages the model was
# (wrongly) confident about outranked real, visible declining pages. Cost/value
# logic belongs in the sort, not just the prose.
action_priority = {
    "PRIORITY_REFRESH_REVIEW": 0,
    "SCHEDULE_REFRESH": 1,
    "REVIEW_SNIPPET_CTR": 2,
    "MONITOR_ONLY": 3,
    "NO_ACTION": 4,
}
queue["action_priority"] = queue["suggested_action"].map(action_priority)
queue = queue.sort_values(
    ["action_priority", "impressions_90d"], ascending=[True, False]
).reset_index(drop=True)
queue["rank"] = queue.index + 1

print("Archetype -> action mapping:")
print(queue.groupby("archetype")["suggested_action"].agg(lambda s: s.iloc[0]).to_string())
print()
print("Action mix across the held-out queue:")
print(queue["suggested_action"].value_counts().to_string())
print()
cols_preview = ["rank", "client_id", "archetype", "suggested_action", "confidence",
                "reason_codes_str", "model_probability", "ctr", "avg_position",
                "impressions_90d", "days_since_last_update"]
print("\nTop 12 ranked items:")
print(queue[cols_preview].head(12).to_string(index=False))


Archetype -> action mapping:
archetype
Quiet / Low Visibility                  MONITOR_ONLY
Stable                                     NO_ACTION
Striking-Distance CTR Gap         REVIEW_SNIPPET_CTR
Visible & Declining          PRIORITY_REFRESH_REVIEW
Visible & Stale                     SCHEDULE_REFRESH

Action mix across the held-out queue:
suggested_action
REVIEW_SNIPPET_CTR         2184
MONITOR_ONLY               1755
NO_ACTION                  1127
SCHEDULE_REFRESH            730
PRIORITY_REFRESH_REVIEW      25


Top 12 ranked items:
 rank         client_id           archetype        suggested_action confidence                                                         reason_codes_str  model_probability  ctr  avg_position  impressions_90d  days_since_last_update
    1 client_4e07408562 Visible & Declining PRIORITY_REFRESH_REVIEW       high MODEL_FLAGGED_DECLINE_RISK|CTR_BELOW_POSITION_EXPECTED|VISIBLE_AND_STALE           0.610626 0.06           1.9             3245                    

**The decay/refresh insight, stated carefully.** The `VISIBLE_AND_STALE` archetype exists
because of what Week 6's error analysis found: the model's most confident *false negatives* were
big, actively-updated pages, while its confident *false positives* were quiet, untouched pages it
mistook for "declining" just because they were stale and small. Directional reading, consistent
with the FlyRank paper's own freshness findings: staleness on a page that still carries real
traffic is a more trustworthy refresh signal than staleness alone — which is why `VISIBLE_AND_STALE`
is its own archetype here rather than folded into the model's flag. This is an **observed pattern
in this dataset**, not a guarantee that refreshing any given page will recover traffic.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use.** A content editor or SEO reviewer opens this queue once a week (or before a
planning sprint) as a **sort order for their own review time** — which pages to look at first,
not which pages to change. It supports one decision: *"of everything I could review this week,
where should I start?"*

**Who should NOT use it as-is.** Anyone without access to the actual live pages (this queue only
has anonymized IDs and metrics — no titles, URLs, or content) can't act on it directly; it has to
hand off to someone who can open the real page.

**Where it stops being valid:**
- **The model signal is weak on its own.** Audited AUC of 0.542 against a 0.540 base rate means
  the model alone is close to a coin flip — it earns its place in this queue by being combined
  with the transparent CTR-gap rule, not by being trusted alone.
- **Client-held-out, not universally validated.** The queue above was scored on clients the model
  never trained on, which is the honest test *for this dataset's 31 clients*. It says nothing
  about a client from a very different industry, language, or content type.
- **Cross-sectional, not causal.** Nothing here was tested with an experiment. "This page is
  flagged" is an observed pattern, not a claim that refreshing it *will* recover traffic — see
  `skills/writing-honest-claims/SKILL.md`'s claim ladder.
- **Snapshot, not live.** The underlying data is a single anonymized export; scores go stale as
  soon as real traffic moves (see Section 4).


In [13]:
# A concrete boundary check: how thin is the evidence behind the "high confidence" tier?
print(queue["confidence"].value_counts())
print()
print("Base rate this queue is being compared against:", round(queue["is_declining_label"].mean(), 3))
print("Model AUC on this exact queue:", auc_check)
print("--> 'high' confidence means two independent, weak-to-modest signals agree, not that either")
print("    signal is individually strong. State that plainly whenever this queue is shared.")


confidence
medium    4669
low       1127
high        25
Name: count, dtype: int64

Base rate this queue is being compared against: 0.54
Model AUC on this exact queue: 0.542
--> 'high' confidence means two independent, weak-to-modest signals agree, not that either
    signal is individually strong. State that plainly whenever this queue is shared.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any flagged page, a human must check:**
- Open the real page and confirm it still matches its original intent (the anonymized row can't
  show this).
- Check the current live SERP for that query — the snapshot in this data may already be stale.
- Rule out cannibalization: is another page on the same site already covering this topic better?
- Confirm the page isn't intentionally being sunset, merged, or already mid-edit.
- For `PRIORITY_REFRESH_REVIEW` items specifically: sanity-check that the decline isn't explained
  by something outside content quality (a seasonal topic, a algorithm update, a broken tracking
  tag) before spending editorial time on a rewrite.

**What should NOT be automated:**
- **No auto-publishing or auto-editing content** from this queue or any model output.
- **No automated depublishing, redirecting, or pruning** of pages flagged as low-value —
  `MONITOR_ONLY` means "not urgent," never "delete."
- **No client-facing reporting that states or implies causation** ("this model predicts your
  traffic will grow") — only decision-support language, per the honest-claims skill.
- **No fully automated refresh scheduling** that skips human sign-off, even for `high`-confidence
  items.
- **No use of this queue for individual staff performance evaluation** — it scores pages, not
  people, and the model's own error analysis (Week 6) shows it gets specific pages wrong often
  enough that it would be an unfair yardstick for a person's work.


In [14]:
# Make the no-go list something the notebook itself enforces, not just prose:
# assert that nothing in the queue implies an automatic action was taken.
assert "auto_applied" not in queue.columns, "This queue must never carry an 'auto-applied' flag."
NO_GO_ACTIONS = {"AUTO_PUBLISH", "AUTO_DEPUBLISH", "AUTO_REDIRECT"}
assert not NO_GO_ACTIONS & set(queue["suggested_action"].unique()), (
    "Suggested actions must stay review-only -- none of these should ever appear."
)
print("No-go check passed: every suggested_action requires a human step, none are auto-executing.")
print("Allowed actions:", sorted(queue["suggested_action"].unique()))


No-go check passed: every suggested_action requires a human step, none are auto-executing.
Allowed actions: ['MONITOR_ONLY', 'NO_ACTION', 'PRIORITY_REFRESH_REVIEW', 'REVIEW_SNIPPET_CTR', 'SCHEDULE_REFRESH']


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Concrete, checkable triggers — not vague "monitor performance":

1. **Base-rate drift.** If a fresh export's `is_declining_label` rate moves more than ~10
   percentage points from the 0.540 recorded here, the portfolio itself has shifted and every
   threshold in this notebook (medians, quantiles) needs recomputing, not just re-applying.
2. **AUC decay.** If a re-run of the Week-6 client-grouped audit on new data drops AUC
   meaningfully below the 0.542 baseline recorded here, the model is no longer earning its place
   in the queue and should be dropped back to the rule-only baseline until retrained.
3. **New client onboarded.** Any new client should get its own held-out check before being scored
   at all — the grouped-split audit exists specifically because performance on an unseen client
   isn't guaranteed by performance on known ones.
4. **Feature drift.** If the median of `ctr`, `avg_position`, or `days_since_last_update` shifts
   materially from the training thresholds recorded below, the rule-based flags (fit on training
   medians) are stale and should be recomputed before the next run.
5. **Cadence.** Re-run the full audit-and-playbook pipeline quarterly at minimum, or immediately
   after any known tracking/tagging change (GA4 or GSC config changes silently break `ctr`,
   `impressions_90d`, etc.).


In [15]:
# Snapshot the exact numbers a future run should compare itself against.
monitoring_snapshot = {
    "base_rate": round(float(queue["is_declining_label"].mean()), 4),
    "holdout_auc": auc_check,
    "visibility_threshold_impressions_90d": float(visibility_threshold),
    "staleness_threshold_days_since_update": float(staleness_threshold),
    "gap_threshold_ctr_points": float(gap_threshold),
    "n_train_clients": int(train_grouped["client_id"].nunique()),
    "n_test_clients": int(test_grouped["client_id"].nunique()),
}
print(json.dumps(monitoring_snapshot, indent=2))


{
  "base_rate": 0.5399,
  "holdout_auc": 0.542,
  "visibility_threshold_impressions_90d": 928.0,
  "staleness_threshold_days_since_update": 22.0,
  "gap_threshold_ctr_points": 0.4305036524413686,
  "n_train_clients": 24,
  "n_test_clients": 7
}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The ranked-queue CSV is regenerated by this notebook every run and stays out of git (the CI
leak-guard blocks data files anywhere under `work/`) — that's why it lives only in
`work/outputs/`. The monitoring snapshot is a small JSON, not a dataset, so it gets committed:
it's the receipt the paper's numbers should trace back to. The two summary charts go to
`work/figures/`, committed, for direct reuse in the paper.


In [16]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --- 1) The queue CSV (gitignored by design, regenerated on every run) ---
export_cols = ["rank", "content_id", "client_id", "archetype", "suggested_action", "confidence",
               "reason_codes_str", "model_probability", "ctr_gap", "ctr", "avg_position",
               "impressions_90d", "days_since_last_update", "is_declining_label"]
queue_path = "work/outputs/w07_ranked_queue.csv"
queue[export_cols].to_csv(queue_path, index=False)
print("Wrote", queue_path, "-", len(queue), "rows")

# --- 2) The monitoring snapshot JSON (committed -- the paper's receipts) ---
metrics_path = "work/outputs/w07_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(monitoring_snapshot, f, indent=2)
print("Wrote", metrics_path)

# --- 3) Figures (committed, for direct reuse in the paper) ---
fig1, ax1 = plt.subplots(figsize=(7, 4))
queue["suggested_action"].value_counts().plot(kind="barh", ax=ax1, color="#426B69")
ax1.set_title("Suggested action mix (held-out queue, n=%d)" % len(queue))
ax1.set_xlabel("count")
fig1.tight_layout()
fig1.savefig("work/figures/w07_action_mix.png", dpi=150)
plt.close(fig1)

reason_counts = {}
for codes in queue["reason_codes"]:
    for c in codes:
        reason_counts[c] = reason_counts.get(c, 0) + 1
fig2, ax2 = plt.subplots(figsize=(7, 4))
pd.Series(reason_counts).sort_values().plot(kind="barh", ax=ax2, color="#8C6BB1")
ax2.set_title("Reason code frequency (held-out queue)")
ax2.set_xlabel("count")
fig2.tight_layout()
fig2.savefig("work/figures/w07_reason_codes.png", dpi=150)
plt.close(fig2)

print("Wrote work/figures/w07_action_mix.png")
print("Wrote work/figures/w07_reason_codes.png")


Wrote work/outputs/w07_ranked_queue.csv - 5821 rows
Wrote work/outputs/w07_metrics.json
Wrote work/figures/w07_action_mix.png
Wrote work/figures/w07_reason_codes.png


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.